<div class="alert alert-block alert-info">
<b>Number of points for this notebook:</b> 6
<br>
<b>Deadline:</b> November 25, 2025 (Tuesday) 23:59
<br>
<b>In the final submission, please only modify the cell with #YOUR CODE HERE. The safest way is to copy your solution to a fresh notebook. We ONLY use auto-grading. NO manual correction.
<br>
<b>Please only submit one .zip file that contains all files.
</div>

# Exercise 6.2 Recommender system

In this exercise, your task is to design a recommender system.

## Learning goals:
* Practise tuning a neural network model by using different regularization methods.

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import tools
import data

In [ ]:
skip_training = True  # Set this flag to True before validation and submission

In [ ]:
# During evaluation, this cell sets skip_training to True
# skip_training = True

In [4]:
# When running on your own computer, you can specify the data directory by:
# data_dir = tools.select_data_dir('/your/local/data/directory')
data_dir = tools.select_data_dir()

The data directory is ../data


In [5]:
# Select the device for training (use GPU if you have one)
#device = torch.device('cuda:0')
device = torch.device('cpu')

In [6]:
if skip_training:
    # The models are always evaluated on CPU
    device = torch.device("cpu")

## Ratings dataset

We will train the recommender system on the dataset in which element consists of three values:
* `user_id` - id of the user (the smallest user id is 1)
* `item_id` - id of the movie (the smallest item id is 1)
* `rating` - rating given by the user to the item (ratings are integer numbers between 1 and 5.

The recommender system need to predict the rating for any given pair of `user_id` and `item_id`.

We measure the quality of the predicted ratings using the mean-squared error (MSE) loss:
$$
  \frac{1}{N}\sum_{i=1}^N (r_i - \hat{r}_i)^2
$$
where $r_i$ is a real rating and $\hat{r}_i$ is a predicted one.

Note: The predicted rating $\hat{r}_i$ does not have to be an integer number.

In [7]:
trainset = data.RatingsData(root=data_dir, train=True)
testset = data.RatingsData(root=data_dir, train=False)

100.0%


In [13]:
# Print one sample from the dataset
x = trainset[0]
print(x)
print(f'user_id={x[0]}, item_id={x[1]}, rating={x[2]}')

(tensor(1), tensor(1), tensor(5))
user_id=1, item_id=1, rating=5


# Model

You need to design a recommender system model with the API described in the cell below.

Hints on the model architecture:
* You need to use [torch.nn.Embedding](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html?highlight=embedding#torch.nn.Embedding) layer to convert inputs `user_ids` and `item_ids` into reasonable representations. The idea of the embedding layer is that we want to represent similar users with values that are close to each other. The original representation as integers is not good for that. By using the embedding layer, we can learn such useful representations automatically.

### Model tuning

In this exercise, you need to tune the architecture of your model to achieve the best performance on the provided test set. You will notice that overfitting is a severe problem for this data: The model can easily overfit the training set producing poor accuracy on the out-of-training (test) data.

You need to find an optimal combination of the hyperparameters, with some hyperparameters corresponding to the regularization techniques that we studied in the lecture.

The hyperparameters that you are advised to consider:
* Learning rate value and learning rate schedule (decresing the learning rate often has positive effect on the model performance)
* Number of training epochs
* Network size
* Weight decay
* Early stopping
* Dropout
* Increase amount of data:
  * Data augmentation
  * Injecting noise

You can tune the hyperparameters by, for example, grid search, random search or manual tuning.

Note:
* The number of points that you will get from this exercise depends on the MSE loss on the test set:
  * below 1.00: 1 point
  * below 0.96: 2 points
  * below 0.95: 3 points
  * below 0.94: 4 points
  * below 0.93: 5 points
  * below 0.92: 6 points 

In [116]:
class RecommenderSystem(nn.Module):
    def __init__(self, n_users, n_items):
        """
        Args:
          n_users: Number of users.
          n_items: Number of items.
        """
        # YOUR CODE HERE
        super().__init__()
        emb_dim = 64
        self.user_emb = nn.Embedding(n_users + 1, emb_dim, padding_idx=0)
        self.item_emb = nn.Embedding(n_items + 1, emb_dim, padding_idx=0)
        self.user_bias = nn.Embedding(n_users + 1, 1, padding_idx=0)
        self.item_bias = nn.Embedding(n_items + 1, 1, padding_idx=0)
        self.global_bias = nn.Parameter(torch.zeros(1))
        in_dim = 3 * emb_dim
        self.fc1 = nn.Linear(in_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc_out = nn.Linear(64, 1)
        
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(p=0.3)
        for layer in [self.fc1, self.fc2, self.fc_out]:
            nn.init.kaiming_uniform_(layer.weight, nonlinearity="relu")
            nn.init.zeros_(layer.bias)
    
    def add_noise(self,x,std): 
        return x + std
        
    def forward(self, user_ids, item_ids):
        """
        Args:
          user_ids of shape (batch_size): User ids (starting from 1).
          item_ids of shape (batch_size): Item ids (starting from 1).
        
        Returns:
          outputs of shape (batch_size): Predictions of ratings.
        """
        u = self.user_emb(user_ids)   # (B, emb_dim)
        i = self.item_emb(item_ids)   # (B, emb_dim)
        ui = u * i
        x = torch.cat([u, i, ui], dim=-1)  # (B, 3*emb_dim)
        
        h = self.dropout(self.act(self.fc1(x)))
        h = self.dropout(self.act(self.fc2(h)))
        mlp_score = self.fc_out(h).squeeze(-1)  # (B,)
        ub = self.user_bias(user_ids).squeeze(-1)
        ib = self.item_bias(item_ids).squeeze(-1)
        
        pred = mlp_score + ub + ib + self.global_bias
        return pred

        


You can test the shapes of the model outputs using the function below.

In [117]:
def test_RecommenderSystem_shapes():
    n_users, n_items = 100, 1000
    model = RecommenderSystem(n_users, n_items)
    batch_size = 10
    user_ids = torch.arange(1, batch_size+1)
    item_ids = torch.arange(1, batch_size+1)
    output = model(user_ids, item_ids)
    assert output.shape == torch.Size([batch_size]), "Wrong output shape."
    print('Success')

test_RecommenderSystem_shapes()

Success


In [ ]:
# This cell is reserved for testing

## Train the model

You need to train a recommender system using **only the training data.** Please use the test set to select the best model: the model that generalizes best to out-of-training data.

In [118]:
# Create the model
model = RecommenderSystem(trainset.n_users, trainset.n_items)

In [119]:
def add_noise(x, noise_std):
    """Add Gaussian noise to a PyTorch tensor.
    
    Args:
      x (tensor): PyTorch tensor of inputs.
      noise_std (float): Standard deviation of the Gaussian noise.
      
    Returns:
      x: Tensor with Gaussian noise added.
    """
    # YOUR CODE HERE
    return x + noise_std
def compute_loss(model, user,item, y):
    model.eval()
    with torch.no_grad():
        outputs = model(user,item)
        loss = F.mse_loss(outputs, y)
        return loss.cpu().numpy()

In [120]:
# Implement the training loop in this cell
from torch.utils.data import DataLoader

criterion = nn.MSELoss()
batch_size = 2048
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=2e-3,          
    weight_decay=1e-4
)

trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True)

num_epochs = 20

if not skip_training:
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for user_ids, item_ids, ratings in trainloader:
            user_ids = user_ids.to(device)
            item_ids = item_ids.to(device)
            ratings = ratings.to(device).float()

            preds = model(user_ids, item_ids)

            loss = criterion(preds, ratings)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * ratings.size(0)

        epoch_loss = running_loss / len(trainset)
        print(f"Epoch {epoch+1}/{num_epochs} - Train MSE: {epoch_loss:.4f}")

    

Epoch 1/20 - Train MSE: 4.1750
Epoch 2/20 - Train MSE: 2.8541
Epoch 3/20 - Train MSE: 2.3423
Epoch 4/20 - Train MSE: 2.0101
Epoch 5/20 - Train MSE: 1.7688
Epoch 6/20 - Train MSE: 1.5929
Epoch 7/20 - Train MSE: 1.4565
Epoch 8/20 - Train MSE: 1.3585
Epoch 9/20 - Train MSE: 1.2855
Epoch 10/20 - Train MSE: 1.2219
Epoch 11/20 - Train MSE: 1.1729
Epoch 12/20 - Train MSE: 1.1297
Epoch 13/20 - Train MSE: 1.0924
Epoch 14/20 - Train MSE: 1.0619
Epoch 15/20 - Train MSE: 1.0402
Epoch 16/20 - Train MSE: 1.0195
Epoch 17/20 - Train MSE: 1.0037
Epoch 18/20 - Train MSE: 0.9838
Epoch 19/20 - Train MSE: 0.9750
Epoch 20/20 - Train MSE: 0.9601


In [121]:
print(compute_loss(model, testset[:][0], testset[:][1], testset[:][2]))

0.98080957


In [122]:
# Save the model to disk (the pth-files will be submitted automatically together with your notebook)
# Set confirm=False if you do not want to be asked for confirmation before saving.
if not skip_training:
    tools.save_model(model, 'recsys.pth', confirm=True)

Model saved to recsys.pth.


In [ ]:
if skip_training:
    model = RecommenderSystem(trainset.n_users, trainset.n_items)
    tools.load_model(model, 'recsys.pth', device)

In [ ]:
# This cell is reserved for grading

In [ ]:
# This cell is reserved for grading

In [ ]:
# This cell is reserved for grading

In [ ]:
# This cell is reserved for grading

In [ ]:
# This cell is reserved for grading

In [ ]:
# This cell is reserved for grading

In [ ]:
# This cell is reserved for grading